# Bloque 5: Monitoreo y reentrenamiento

Este notebook implementa el bloque 5 del harness. Detecta deriva (drift) en features y predicciones inyectando el mes 13 de drift, calcula PSI (Population Stability Index) por feature, dispara alertas si PSI > 0.3, y define la politica de reentrenamiento diario que entra como challenger al Bloque 3.

In [0]:
# 1. Feature drift: comparar features baseline vs mes 13 (drift inyectado)

import numpy as np
import pandas as pd

# Funcion PSI (Population Stability Index)
def calculate_psi(expected, actual, buckets=10):
    breakpoints = np.percentile(expected, np.linspace(0, 100, buckets + 1))
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf
    expected_counts = np.histogram(expected, bins=breakpoints)[0]
    actual_counts = np.histogram(actual, bins=breakpoints)[0]
    expected_pct = np.where(expected_counts / len(expected) == 0, 0.0001, expected_counts / len(expected))
    actual_pct = np.where(actual_counts / len(actual) == 0, 0.0001, actual_counts / len(actual))
    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return float(psi)

# Cargar features baseline (ultima foto de customer_features)
baseline_pd = spark.sql("""
    SELECT f.*
    FROM andina_market_lhcl_prod.features.customer_features f
    INNER JOIN (
        SELECT customer_id, MAX(feature_timestamp) AS max_ts
        FROM andina_market_lhcl_prod.features.customer_features
        GROUP BY customer_id
    ) latest
      ON f.customer_id = latest.customer_id 
      AND f.feature_timestamp = latest.max_ts
""").toPandas()

print(f"Baseline: {len(baseline_pd)} clientes")

# Computar features del mes 13 desde tablas raw de drift
# Orders features
month13_orders = spark.sql("""
    SELECT 
        customer_id,
        COUNT(*) AS orders_count_30d,
        AVG(total_amount) AS avg_ticket_value_90d,
        DATEDIFF('2026-04-01', MAX(order_ts)) AS days_since_last_order,
        DATEDIFF('2026-04-01', MIN(order_ts)) AS tenure_days
    FROM andina_market_lhcl_prod.rdv_retail.orders_month_13
    GROUP BY customer_id
""").toPandas()

# App events features
month13_events = spark.sql("""
    SELECT 
        customer_id,
        SUM(CASE WHEN event_type = 'view_product' THEN 1 ELSE 0 END) AS view_product_count_14d,
        SUM(CASE WHEN event_type = 'search' THEN 1 ELSE 0 END) AS search_count_14d,
        COUNT(*) AS web_activity_count_14d,
        SUM(CASE WHEN event_type LIKE '%cart%' THEN 1 ELSE 0 END) AS cart_abandonment_sessions_14d
    FROM andina_market_lhcl_prod.rdv_retail.app_events_month_13
    GROUP BY customer_id
""").toPandas()

# Merge month 13 features
month13_features = month13_orders.merge(month13_events, on='customer_id', how='outer').fillna(0)
# orders_count_90d = same as 30d for one month of data
month13_features['orders_count_90d'] = month13_features['orders_count_30d']
# Defaults for features we can't compute from raw
month13_features['payment_failure_rate_90d'] = 0.0
month13_features['favorite_product_sku'] = 'UNKNOWN'
month13_features['customer_segment'] = 'UNKNOWN'

print(f"Mes 13 (drift): {len(month13_features)} clientes")

# Comparar features con PSI
NUMERIC_FEATURES = ['days_since_last_order', 'orders_count_30d', 'orders_count_90d',
                    'avg_ticket_value_90d', 'payment_failure_rate_90d',
                    'view_product_count_14d', 'search_count_14d', 'web_activity_count_14d',
                    'cart_abandonment_sessions_14d', 'tenure_days']

drift_results = []
for feature in NUMERIC_FEATURES:
    if feature in month13_features.columns and feature in baseline_pd.columns:
        baseline_values = baseline_pd[feature].dropna().values
        drift_values = month13_features[feature].dropna().values
        if len(drift_values) > 0 and len(baseline_values) > 0:
            psi = calculate_psi(baseline_values, drift_values)
            status = "DRIFT DETECTADO" if psi > 0.3 else "OK"
            drift_results.append({
                'feature': feature,
                'psi': round(psi, 4),
                'status': status,
                'baseline_mean': round(float(baseline_values.mean()), 2),
                'drift_mean': round(float(drift_values.mean()), 2),
                'cambio_pct': round(float((drift_values.mean() - baseline_values.mean()) / max(abs(baseline_values.mean()), 0.001) * 100), 1)
            })

drift_df = pd.DataFrame(drift_results).sort_values('psi', ascending=False)
print("\n=== Feature Drift Detection (PSI > 0.3 = DRIFT) ===")
print(f"Umbral: PSI > 0.3 | Alerta a: consumidores del reporte | Canal: Slack/Teams\n")
display(drift_df)

drifted_features = drift_df[drift_df['status'] == 'DRIFT DETECTADO']
print(f"\nFeatures con drift: {len(drifted_features)} de {len(drift_df)}")
if len(drifted_features) > 0:
    print("Reporte rapido para Slack/Teams:")
    for _, row in drifted_features.iterrows():
        print(f"  - {row['feature']}: PSI={row['psi']} | Baseline mean={row['baseline_mean']} -> Drift mean={row['drift_mean']} ({row['cambio_pct']}%)")

In [0]:
# 2. Prediction drift: scorear mes 13 con champion y comparar distribucion de predicciones

import mlflow
import mlflow.sklearn

MODEL_FULL_NAME = "workspace.modelos_prod.repurchase_propensity"
CHAMPION_URI = f"models:/{MODEL_FULL_NAME}@champion"

mlflow.set_registry_uri("databricks-uc")

# Cargar champion
model = mlflow.sklearn.load_model(CHAMPION_URI)

# Features que el modelo espera
FEATURE_COLUMNS = ['days_since_last_order', 'orders_count_30d', 'orders_count_90d',
                    'avg_ticket_value_90d', 'payment_failure_rate_90d',
                    'view_product_count_14d', 'search_count_14d', 'web_activity_count_14d',
                    'cart_abandonment_sessions_14d', 'tenure_days',
                    'favorite_product_sku', 'customer_segment']

# Scorear baseline (predicciones normales)
baseline_pred = model.predict_proba(baseline_pd[FEATURE_COLUMNS])[:, 1]

# Scorear mes 13 (predicciones con drift)
drift_pred = model.predict_proba(month13_features[FEATURE_COLUMNS])[:, 1]

# PSI de predicciones
psi_predictions = calculate_psi(baseline_pred, drift_pred)
print("=== Prediction Drift ===")
print(f"Baseline: propension promedio = {baseline_pred.mean():.4f}")
print(f"Mes 13:   propension promedio = {drift_pred.mean():.4f}")
print(f"PSI de predicciones: {psi_predictions:.4f}")
print(f"Status: {'DRIFT DETECTADO' if psi_predictions > 0.3 else 'OK'}")

# Comparacion de distribuciones
print("\n=== Distribucion de predicciones ===")
comparison = pd.DataFrame({
    'bucket': ['Muy baja (<0.2)', 'Baja (0.2-0.4)', 'Media (0.4-0.6)', 'Alta (0.6-0.8)', 'Muy alta (>=0.8)'],
    'baseline_pct': [
        (baseline_pred < 0.2).mean(),
        ((baseline_pred >= 0.2) & (baseline_pred < 0.4)).mean(),
        ((baseline_pred >= 0.4) & (baseline_pred < 0.6)).mean(),
        ((baseline_pred >= 0.6) & (baseline_pred < 0.8)).mean(),
        (baseline_pred >= 0.8).mean(),
    ],
    'drift_pct': [
        (drift_pred < 0.2).mean(),
        ((drift_pred >= 0.2) & (drift_pred < 0.4)).mean(),
        ((drift_pred >= 0.4) & (drift_pred < 0.6)).mean(),
        ((drift_pred >= 0.6) & (drift_pred < 0.8)).mean(),
        (drift_pred >= 0.8).mean(),
    ]
})
comparison['cambio_pct'] = ((comparison['drift_pct'] - comparison['baseline_pct']) * 100).round(1)
display(comparison)

print("\nConclusion: el drift en features causa drift en predicciones.")
print("El modelo produce predicciones diferentes porque la data de entrada cambio.")

## 3. Label drift + Alerta + Politica de reentrenamiento

### Label drift (deriva de etiqueta)
- La etiqueta es `repurchased_next_30d` (0 = no recompro, 1 = recompro).
- Label drift = la proporcion de clientes que recompran cambio respecto al entrenamiento.
- **Solo se detecta 30 dias despues** de la prediccion (hay que esperar a ver si el cliente recompro).
- En el entrenamiento: 58% recompraban. Si en el mes 13 la proporcion baja a 35%, hay label drift.
- Implementacion: comparar la tasa de recompra real (cuando llegue la etiqueta) con el baseline.

### Alerta (PSI > 0.3)
- **Quien se entera:** Los consumidores del reporte del modelo (marketing, analytics).
- **Canal:** Slack / Teams / chat.
- **Contenido del reporte:** Feature que driftio, PSI, baseline mean vs drift mean, % de cambio, numero de clientes afectados.
- **Trigger:** PSI > 0.3 en cualquier feature o en predicciones.

### Politica de reentrenamiento
- **Frecuencia:** Diaria por calendario (Lakeflow Jobs).
- **Flujo del reentrenamiento:**
  1. Reentrenar modelo con data actualizada
  2. Registrar nueva version en UC
  3. Asignar `@challenger` a la nueva version
  4. Evaluar vs `@champion` en mismo test set (criterios del Bloque 3)
  5. Si pasa criterios -> promover moviendo `@champion`
  6. **Nunca directo a produccion**
- **Reentrenamiento extra por alerta:** Si PSI > 0.3, reentrenar inmediatamente ademas del calendario diario.

## 4. Extension de diseño: ambientes dev / test / prod

### Diagrama del ciclo entre ambientes

```
┌─────────────────────────────────────────────────────────────┐
│  DEV (desarrollo)                                           │
│  - Notebook de feature engineering + entrenamiento           │
│  - Experimentacion con hiperparametros                      │
│  - MLflow experiments locales                                │
│  - Datos: muestra o datos historicos                        │
│  - Sin job programado, sin serving                          │
└──────────────────────┬──────────────────────────────────────┘
                       │ Pull request + code review (aprobacion humana)
┌──────────────────────▼──────────────────────────────────────┐
│  TEST (pruebas)                                              │
│  - CI/CD: correr notebooks automaticamente                   │
│  - Validar metricas contra baseline ( umbrales)              │
│  - Test de integracion: feature pipeline + scoring           │
│  - Datos: datos de prueba completos                          │
│  - Job de prueba (no productivo)                             │
└──────────────────────┬──────────────────────────────────────┘
                       │ Aprobacion humana (promover a prod)
┌──────────────────────▼──────────────────────────────────────┐
│  PROD (produccion)                                           │
│  - Job batch diario programado                              │
│  - Endpoint real-time (serving)                              │
│  - Monitor de drift (PSI > 0.3)                              │
│  - Inference tables / batch_predictions                     │
│  - Alertas a Slack/Teams                                     │
│  - Reentrenamiento diario -> challenger -> promocion         │
└─────────────────────────────────────────────────────────────┘
```

### Promover modelo vs promover codigo

| Enfoque | Que se mueve | Ventaja | Desventaja |
| --- | --- | --- | --- |
| **Promover modelo** | Registrar modelo en UC, cargar por alias | Rapido, sin reentrenar | Si los datos cambian, el modelo envejece |
| **Promover codigo** | Llevar el notebook de entrenamiento al siguiente ambiente y reentrenar | Modelo se adapta a datos locales | Mas lento, requiere compute en cada ambiente |

**Recomendacion para Andina Market:** Promover codigo entre dev -> test, y promover modelo entre test -> prod. En test, reentrenar y validar. En prod, cargar el modelo validado por alias.

### Que empaquetar como Bundle (DAB)
- Notebooks de feature engineering, entrenamiento, scoring, monitoreo
- Job de inferencia batch (programado diario)
- Job de reentrenamiento (programado diario)
- Alertas SQL (drift detection)
- Configuracion del endpoint (en produccion)

### Donde interviene aprobacion humana
1. **dev -> test:** Pull request + code review
2. **test -> prod:** Aprobacion manual de metricas (Precision@Top-20% > baseline) + revision de drift
3. **Promocion de modelo:** El data scientist mueve `@champion` manualmente despues de revisar el challenger

### CI/CD
- Git: notebooks versionados en Databricks Repos
- CI: al hacer push, correr notebooks de test automaticamente
- CD: al aprobar, deployar el bundle al ambiente de produccion
- DAB (Declarative Automation Bundles): `databricks bundle deploy` mueve todo el proyecto entre ambientes

## README — Bloque 5: Monitoreo y reentrenamiento

### Objetivo
Detectar deriva (drift) en features y predicciones, definir que pasa cuando ocurre, y disenar el ciclo entre ambientes (dev/test/prod).

### Decisiones tomadas (Decide tu)
1. **Que monitorear:** Los 3 tipos de drift (feature, prediction, label). Feature drift es el mas comun y se detecta en tiempo real. Label drift tarda 30 dias.
2. **Umbrales y alerta:** PSI > 0.3. Se informan a los consumidores del reporte del modelo via Slack/Teams. Reporte rapido con feature, PSI, baseline mean, drift mean y % de cambio.
3. **Politica de reentrenamiento:** Diaria por calendario (Lakeflow Jobs). El modelo reentrenado entra como `@challenger` al Bloque 3, nunca directo a produccion. Reentrenamiento extra si PSI > 0.3.

### Demostracion (mes 13 inyectado)
1. Se cargan features del mes 13 desde tablas raw de drift (`orders_month_13`, `app_events_month_13`).
2. Se calcula PSI por feature comparando baseline vs mes 13.
3. Features con PSI > 0.3 se marcan como DRIFT DETECTADO.
4. Se scorea el mes 13 con `@champion` y se compara la distribucion de predicciones.
5. El monitor detecta el drift antes de que el negocio lo reporte.

### Tipos de drift implementados
| Tipo | Como se detecta | Cuando |
| --- | --- | --- |
| Feature drift | PSI por feature > 0.3 | Tiempo real (al scorear) |
| Prediction drift | PSI de predicciones > 0.3 | Tiempo real (al scorear) |
| Label drift | Comparar tasa de recompra real vs baseline | 30 dias despues |

### Extension de diseño (dev/test/prod)
- Promover codigo entre dev -> test (PR + code review).
- Promover modelo entre test -> prod (UC alias @champion).
- DAB empaqueta notebooks, jobs, alertas.
- Aprobacion humana en promocion de modelo y en paso a prod.

### Evidencias a capturar (manual)
1. Screenshot de la tabla de feature drift (PSI por feature).
2. Screenshot de la tabla de prediction drift (distribucion baseline vs drift).
3. Guardar en `/harness_config/evidence/block5/`.

### Que esperamos que domines

**Pregunta 1:** Que es Data Profiling (antes Lakehouse Monitoring)?
- Perfiles de inferencia, metricas de drift, ventanas de tiempo y slicing en Data Quality Monitoring de UC. Aqui lo implementamos con PSI en Python.

**Pregunta 2:** Como configuras una alerta de Databricks SQL?
- Creas una query SQL que evalua `PSI > 0.3` y configuras una alerta que envia email/Slack cuando se cumple.

**Pregunta 3:** Que es "desplegar codigo" vs "desplegar modelo"?
- Desplegar codigo = llevar notebooks al siguiente ambiente y reentrenar. Desplegar modelo = registrar en UC y cargar por alias.

**Pregunta 4:** Que es un DAB (Declarative Automation Bundle)?
- Define todo el proyecto (notebooks, jobs, alertas) como codigo YAML, versionable en Git, deployable entre ambientes.

**Pregunta 5:** Como entra el modelo reentrenado al Bloque 3?
- Como `@challenger`. Se evalua vs `@champion`. Si pasa los criterios, se promueve. Nunca directo a produccion.